In [2]:
# CMS DE-SynPUF Data Verification
# Run this notebook to verify all data files are loaded correctly

import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("CMS DE-SynPUF DATA VERIFICATION")
print("="*70)

# Define data paths
data_path = Path('../data/raw/cms_synpuf/')

# Files to load
files = {
    'inpatient': 'Inpatient_Claims.csv',
    'outpatient': 'Outpatient_Claims.csv',
    'bene_2008': 'Beneficiary_2008.csv',
    'bene_2009': 'Beneficiary_2009.csv',
    'bene_2010': 'Beneficiary_2010.csv'
}

# Check if files exist
print("\n1️⃣ CHECKING FILE AVAILABILITY:")
print("-" * 70)
for key, filename in files.items():
    filepath = data_path / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f" {key:15} | {size_mb:6.1f} MB | {filename}")
    else:
        print(f" {key:15} | MISSING | {filename}")

# Load all datasets
print("\n\n2️⃣ LOADING DATASETS:")
print("-" * 70)

try:
    inpatient = pd.read_csv(data_path / files['inpatient'])
    print(f" Inpatient Claims loaded: {inpatient.shape[0]:,} rows × {inpatient.shape[1]} columns")
except Exception as e:
    print(f" Error loading inpatient: {e}")
    inpatient = None

try:
    outpatient = pd.read_csv(data_path / files['outpatient'])
    print(f" Outpatient Claims loaded: {outpatient.shape[0]:,} rows × {outpatient.shape[1]} columns")
except Exception as e:
    print(f" Error loading outpatient: {e}")
    outpatient = None

try:
    bene_2008 = pd.read_csv(data_path / files['bene_2008'])
    print(f" Beneficiary 2008 loaded: {bene_2008.shape[0]:,} rows × {bene_2008.shape[1]} columns")
except Exception as e:
    print(f" Error loading bene_2008: {e}")
    bene_2008 = None

try:
    bene_2009 = pd.read_csv(data_path / files['bene_2009'])
    print(f" Beneficiary 2009 loaded: {bene_2009.shape[0]:,} rows × {bene_2009.shape[1]} columns")
except Exception as e:
    print(f" Error loading bene_2009: {e}")
    bene_2009 = None

try:
    bene_2010 = pd.read_csv(data_path / files['bene_2010'])
    print(f" Beneficiary 2010 loaded: {bene_2010.shape[0]:,} rows × {bene_2010.shape[1]} columns")
except Exception as e:
    print(f" Error loading bene_2010: {e}")
    bene_2010 = None

# Dataset summaries
if inpatient is not None:
    print("\n\n3️⃣ INPATIENT CLAIMS SUMMARY:")
    print("-" * 70)
    print(f"Total Claims: {len(inpatient):,}")
    print(f"Unique Beneficiaries: {inpatient['DESYNPUF_ID'].nunique():,}")
    print(f"Unique Providers: {inpatient['PRVDR_NUM'].nunique():,}")
    print(f"Date Range: {inpatient['CLM_FROM_DT'].min()} to {inpatient['CLM_THRU_DT'].max()}")
    print(f"\nTotal Payment Amount: ${inpatient['CLM_PMT_AMT'].sum():,.0f}")
    print(f"Average Payment per Claim: ${inpatient['CLM_PMT_AMT'].mean():,.2f}")
    print(f"Median Payment per Claim: ${inpatient['CLM_PMT_AMT'].median():,.2f}")
    print(f"\nMissing Values:")
    missing = inpatient.isnull().sum()
    missing_pct = (missing / len(inpatient) * 100).round(1)
    missing_summary = pd.DataFrame({
        'Missing Count': missing[missing > 0],
        'Missing %': missing_pct[missing > 0]
    }).sort_values('Missing %', ascending=False).head(10)
    print(missing_summary)

if outpatient is not None:
    print("\n\n4️⃣ OUTPATIENT CLAIMS SUMMARY:")
    print("-" * 70)
    print(f"Total Claims: {len(outpatient):,}")
    print(f"Unique Beneficiaries: {outpatient['DESYNPUF_ID'].nunique():,}")
    print(f"Unique Providers: {outpatient['PRVDR_NUM'].nunique():,}")
    print(f"Date Range: {outpatient['CLM_FROM_DT'].min()} to {outpatient['CLM_THRU_DT'].max()}")
    print(f"\nTotal Payment Amount: ${outpatient['CLM_PMT_AMT'].sum():,.0f}")
    print(f"Average Payment per Claim: ${outpatient['CLM_PMT_AMT'].mean():,.2f}")
    print(f"Median Payment per Claim: ${outpatient['CLM_PMT_AMT'].median():,.2f}")

if all([bene_2008 is not None, bene_2009 is not None, bene_2010 is not None]):
    print("\n\n5️⃣ BENEFICIARY DATA SUMMARY:")
    print("-" * 70)
    print(f"2008 Beneficiaries: {len(bene_2008):,}")
    print(f"2009 Beneficiaries: {len(bene_2009):,}")
    print(f"2010 Beneficiaries: {len(bene_2010):,}")
    
    # Check for duplicates across years
    all_bene_ids = pd.concat([
        bene_2008['DESYNPUF_ID'],
        bene_2009['DESYNPUF_ID'],
        bene_2010['DESYNPUF_ID']
    ])
    unique_bene = all_bene_ids.nunique()
    print(f"Unique Beneficiaries (across all years): {unique_bene:,}")
    
    # Chronic conditions prevalence (2010 data as example)
    print("\n Chronic Conditions Prevalence (2010):")
    conditions = {
        'Alzheimer\'s Disease': 'SP_ALZHDMTA',
        'Congestive Heart Failure': 'SP_CHF',
        'Chronic Kidney Disease': 'SP_CHRNKIDN',
        'Cancer': 'SP_CNCR',
        'COPD': 'SP_COPD',
        'Depression': 'SP_DEPRESSN',
        'Diabetes': 'SP_DIABETES',
        'Ischemic Heart Disease': 'SP_ISCHMCHT',
        'Osteoporosis': 'SP_OSTEOPRS',
        'Rheumatoid Arthritis': 'SP_RA_OA',
        'Stroke': 'SP_STRKETIA'
    }
    
    for condition, col in conditions.items():
        prevalence = (bene_2010[col] == 1).sum() / len(bene_2010) * 100
        print(f"  {condition:30} {prevalence:5.1f}%")

# Data quality checks
print("\n\n6️⃣ DATA QUALITY CHECKS:")
print("-" * 70)

if inpatient is not None:
    # Check for negative payment amounts
    negative_payments = (inpatient['CLM_PMT_AMT'] < 0).sum()
    print(f"Inpatient - Negative payment amounts: {negative_payments} ({negative_payments/len(inpatient)*100:.2f}%)")
    
    # Check for zero payment amounts
    zero_payments = (inpatient['CLM_PMT_AMT'] == 0).sum()
    print(f"Inpatient - Zero payment amounts: {zero_payments} ({zero_payments/len(inpatient)*100:.2f}%)")
    
    # Check for duplicate claim IDs
    duplicates = inpatient['CLM_ID'].duplicated().sum()
    print(f"Inpatient - Duplicate claim IDs: {duplicates}")

if outpatient is not None:
    negative_payments = (outpatient['CLM_PMT_AMT'] < 0).sum()
    print(f"Outpatient - Negative payment amounts: {negative_payments} ({negative_payments/len(outpatient)*100:.2f}%)")
    
    zero_payments = (outpatient['CLM_PMT_AMT'] == 0).sum()
    print(f"Outpatient - Zero payment amounts: {zero_payments} ({zero_payments/len(outpatient)*100:.2f}%)")
    
    duplicates = outpatient['CLM_ID'].duplicated().sum()
    print(f"Outpatient - Duplicate claim IDs: {duplicates}")

# Next steps
print("\n\n7️⃣ NEXT STEPS:")
print("-" * 70)
print("All data files loaded successfully!")
print("\nRecommended workflow:")
print("1. Run notebook: 01_eda_initial.ipynb - Exploratory Data Analysis")
print("2. Run notebook: 02_data_preprocessing.ipynb - Clean and prepare data")
print("3. Run notebook: 03_feature_engineering.ipynb - Create ML features")
print("4. Run notebook: 04_fraud_modeling.ipynb - Build fraud detection models")

print("\n" + "="*70)
print("DATA VERIFICATION COMPLETE")
print("="*70)

CMS DE-SynPUF DATA VERIFICATION

1️⃣ CHECKING FILE AVAILABILITY:
----------------------------------------------------------------------
 inpatient       |   15.9 MB | Inpatient_Claims.csv
 outpatient      |  154.3 MB | Outpatient_Claims.csv
 bene_2008       |   13.9 MB | Beneficiary_2008.csv
 bene_2009       |   13.8 MB | Beneficiary_2009.csv
 bene_2010       |   13.4 MB | Beneficiary_2010.csv


2️⃣ LOADING DATASETS:
----------------------------------------------------------------------
 Inpatient Claims loaded: 66,773 rows × 81 columns
 Outpatient Claims loaded: 790,790 rows × 76 columns
 Beneficiary 2008 loaded: 116,352 rows × 32 columns
 Beneficiary 2009 loaded: 114,538 rows × 32 columns
 Beneficiary 2010 loaded: 112,754 rows × 32 columns


3️⃣ INPATIENT CLAIMS SUMMARY:
----------------------------------------------------------------------
Total Claims: 66,773
Unique Beneficiaries: 37,780
Unique Providers: 2,675
Date Range: 20071127.0 to 20101231.0

Total Payment Amount: $639,260,18